# Problem 3. 패션 아이템 이미지 분류하기

## 문제 개요
온라인 쇼핑몰은 판매자가 올린 상품 사진을 **10개 카테고리**로 자동 분류하려 합니다.
28×28 흑백 상품 이미지로 카테고리를 예측하세요.

## 파일 구조
```
problem03/
├── work.ipynb      ← 이 노트북
├── submission.csv        ← 제출 파일 (직접 생성)
└── data/
    ├── train.csv
    ├── test.csv
    └── sample_submission.csv
```

## 데이터
- `data/train.csv` : 12,000장 (label 포함)
- `data/test.csv` : 5,000장 (label 없음)

| 컬럼 | 설명 |
|---|---|
| id | 고유 번호 |
| **label** | **카테고리 (예측 대상)** |
| pixel0 ~ pixel783 | 28×28 이미지를 한 줄로 편 픽셀 밝기(0~255). `pixel{28*행 + 열}` |

| label | 카테고리 | label | 카테고리 |
|---|---|---|---|
| 0 | 티셔츠/탑 | 5 | 샌들 |
| 1 | 바지 | 6 | 셔츠 |
| 2 | 풀오버 | 7 | 스니커즈 |
| 3 | 드레스 | 8 | 가방 |
| 4 | 코트 | 9 | 앵클부츠 |

### 데이터 특징
- 카테고리별 개수는 거의 균등합니다.
- 티셔츠·셔츠·풀오버·코트처럼 **모양이 비슷한 상의**를 구분하는 것이 핵심 난관입니다.
- 출처: Fashion-MNIST (Zalando Research, MIT License) 일부를 대회용으로 재구성

## 규칙
- 제한 시간: 3문제 합계 **90분**
- 허용: pandas, scikit-learn, LightGBM, XGBoost, OpenCV, PyTorch 등 일반 라이브러리와 공개 사전학습 모델
- 금지: LLM API로 테스트 데이터를 직접 예측, 테스트 데이터 수작업 라벨링, 원본 공개 데이터셋에서 정답 조회
- 결과는 이 노트북과 같은 위치(`problem03/`)에 **`submission.csv`** 로 저장

## 평가
**Accuracy (정확도)** — 높을수록 좋습니다.

## 제출 형식
`submission.csv`

| id | label |
|---|---|
| 12000 | 9 |
| 12001 | 2 |

- 컬럼명 `id`, `label`, label은 0~9 정수
- 포맷이 틀리면 0점 처리됩니다.

## 제출 검증 규칙
- 테스트의 모든 ID를 정확히 한 번씩 제출합니다. 누락·중복·추가 ID는 오류이며, 행 순서는 자유입니다(채점 시 ID로 정렬합니다).
- CSV는 UTF-8(또는 UTF-8 BOM), 쉼표 구분, 헤더 포함 형식입니다. 아래 두 컬럼만 제출하며 컬럼 순서는 자유입니다.
- 비어 있는 셀, NaN, 무한대, 깨진 CSV는 오류입니다. 올바르지 않은 제출은 해당 문제 0점과 ERROR로 기록합니다.
- 최종 파일은 이 문제 폴더의 `submission.csv`입니다. 다른 이름이나 하위 폴더 파일은 수집하지 않습니다.
- 컬럼: `id,label`; label은 0~9 정수입니다. 확률 대신 최종 클래스를 제출하세요.
- 행 수: 5,000.
- id는 12000~16999의 정수 ID입니다.

## 100점 환산
`score = 100 × Accuracy` (모든 테스트 행에서 정확히 일치한 클래스의 비율). 세 문제 합계는 300점입니다.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
DATA_DIR = Path("data")
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
print("train:", train.shape, "test:", test.shape)
train.head()

## TODO
데이터를 확인하고 전처리·모델·검증 방법을 구현하세요. 아래는 원문에서 제공한 간단한 베이스라인입니다. 필요에 맞게 수정하세요.

In [ ]:
import os, time, torch, torch.nn as nn, torch.nn.functional as F
from sklearn.model_selection import train_test_split
torch.set_num_threads(os.cpu_count() or 4)

pix = [c for c in train.columns if c.startswith("pixel")]
assert len(pix) == 784 and [c for c in test.columns if c.startswith("pixel")] == pix
to_img = lambda df: torch.tensor(df[pix].values, dtype=torch.float32).view(-1, 1, 28, 28) / 255.
X, y, X_test = to_img(train), torch.tensor(train.label.values), to_img(test)
mu, sd = X.mean(), X.std(); X, X_test = (X - mu) / sd, (X_test - mu) / sd      # train 통계로 표준화

def block(i, o):
    return nn.Sequential(nn.Conv2d(i, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(),
                         nn.Conv2d(o, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(), nn.MaxPool2d(2))
def net(w=32):
    return nn.Sequential(block(1, w), block(w, 2 * w), block(2 * w, 4 * w), nn.Flatten(), nn.Dropout(0.3),
                         nn.Linear(4 * w * 9, 128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, 10))

def augment(xb):                                                            # 좌우반전 + 최대 2픽셀 이동
    flip = torch.rand(xb.size(0)) < 0.5
    xb = torch.where(flip.view(-1, 1, 1, 1), xb.flip(3), xb)
    dx, dy = np.random.randint(-2, 3, 2)
    return torch.roll(xb, (int(dy), int(dx)), (2, 3))

def train_model(Xt, yt, epochs=12, seed=0, bs=128, lr=3e-3):
    torch.manual_seed(seed); np.random.seed(seed); m = net()
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=5e-4)
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * ((len(Xt) + bs - 1) // bs))
    for ep in range(epochs):
        m.train(); perm = torch.randperm(len(Xt))
        for i in range(0, len(Xt), bs):
            idx = perm[i:i + bs]
            loss = F.cross_entropy(m(augment(Xt[idx])), yt[idx], label_smoothing=0.05)
            opt.zero_grad(); loss.backward(); opt.step(); sch.step()
    return m

@torch.no_grad()
def predict_proba(m, Xe):                                                   # 좌우반전 TTA
    m.eval(); out = []
    for i in range(0, len(Xe), 1000):
        xb = Xe[i:i + 1000]; out.append((F.softmax(m(xb), 1) + F.softmax(m(xb.flip(3)), 1)) / 2)
    return torch.cat(out)

EPOCHS, SEEDS = 20, 2          # 홀드아웃 비교(2,400장): 12에폭 0.9117 -> 20에폭 0.9167 (시드 0). 에폭당 약 12초(CPU), 최종 학습 약 8분
RUN_HOLDOUT = False           # True 면 층화 홀드아웃 20% 로 정확도를 다시 측정(약 +4분). 측정값: 시드0/20에폭 0.9167, 시드1/12에폭 0.9062
# ---- 검증(옵션): 층화 홀드아웃 20% (정확도) ----
if RUN_HOLDOUT:
    a, b = train_test_split(np.arange(len(X)), test_size=0.2, stratify=y.numpy(), random_state=42)
    t0 = time.time(); m = train_model(X[a], y[a], EPOCHS, seed=0)
    hold_acc = float((predict_proba(m, X[b]).argmax(1) == y[b]).float().mean())
    print(f"holdout acc ({EPOCHS} epochs, {len(a)} train): {hold_acc:.4f} | {time.time() - t0:.0f}s -> 환산 점수 약 {100 * hold_acc:.1f}")


# ---- 전체 train 으로 재학습한 CNN SEEDS개 확률 평균 ----
t0 = time.time()
probs = [predict_proba(train_model(X, y, EPOCHS, seed=s), X_test) for s in range(SEEDS)]
pred = torch.stack(probs).mean(0).argmax(1).numpy()
print(f"final train+predict: {time.time() - t0:.0f}s | test 예측 분포:", np.bincount(pred, minlength=10).tolist())

In [ ]:
submission = pd.DataFrame({"id": test["id"], "label": pred.astype(int)})
submission.to_csv("submission.csv", index=False)

In [ ]:
submission = pd.read_csv("submission.csv")
print(submission.shape)
assert len(submission) == len(test)
assert submission["id"].is_unique
assert set(submission["id"]) == set(test["id"])
submission.head()
assert list(submission.columns) == ["id", "label"] and submission.label.between(0, 9).all() and submission.label.dtype.kind == "i"